# Lab 2 · Advanced RAG (free Kaggle T4)

**Layer 3 · Data — Chapter 2 (the advanced pipeline).** Lab 1 built naive RAG. Here
we add the four levers that make retrieval *good*, and **measure** that they help:

1. **Metadata filtering** — restrict the search by tag (topic / tenant / date)
2. **Hybrid search** — fuse vector (meaning) with BM25 (exact terms)
3. **Reranking** — a cross-encoder re-scores query+chunk together
4. **Query understanding** — LLM rewrite + multi-query so phrasing doesn't matter

The measured core (levers 1–3 + evaluation) is **CPU-only and runs on any GPU**. The
LLM answer (lever 4 + grounded generation) is the last cell and needs a **T4** runtime.

**Settings:** Accelerator = **GPU T4 x2** (not P100), Internet = **On**, **Run All**.

*Data: [`ItshMoh/kubernetes_qa_pairs`](https://huggingface.co/datasets/ItshMoh/kubernetes_qa_pairs).*

In [ ]:
import warnings, logging, os
warnings.filterwarnings('ignore')
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['HF_HUB_VERBOSITY'] = 'error'
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
for _n in ('huggingface_hub', 'sentence_transformers', 'transformers', 'datasets'):
    logging.getLogger(_n).setLevel(logging.ERROR)

!pip install -q faiss-cpu sentence-transformers datasets rank-bm25 2>/dev/null

import faiss, textwrap
import numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer, CrossEncoder

## 0 · Rebuild the index (recap of Lab 1)

Pull the dataset, embed every answer with **`bge-small`** (CPU), load a flat **FAISS** index. Self-contained — no dependency on Lab 1.

In [ ]:
from datasets import load_dataset
ds = load_dataset('ItshMoh/kubernetes_qa_pairs', split='train')
df = ds.to_pandas()

EMB_ID = 'BAAI/bge-small-en-v1.5'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
embedder = SentenceTransformer(EMB_ID, device='cpu')

chunks = [{'id': f'qa-{i}', 'text': r['answer'], 'question': r['question'], 'topic': r['topic']}
          for i, r in enumerate(ds) if (r['answer'] or '').strip()]
vecs = embedder.encode([c['text'] for c in chunks], normalize_embeddings=True,
                       convert_to_numpy=True, batch_size=128).astype('float32')
index = faiss.IndexFlatIP(vecs.shape[1]); index.add(vecs)

def retrieve(query, k=3):
    qv = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True,
                         convert_to_numpy=True).astype('float32')
    sc, ii = index.search(qv, k)
    return [{**chunks[int(i)], 'score': float(s)} for s, i in zip(sc[0], ii[0])]

print('indexed', index.ntotal, 'answer-chunks |', df['topic'].nunique(), 'topics')

## 1 · Metadata filtering

Every chunk carries metadata (here: `topic`). Filtering at retrieval enforces *permissions* and *freshness* in production (only this tenant, only since 2026). Flat FAISS has no native filter, so we over-fetch and filter in Python (a real vector DB does this inside the search).

In [ ]:
from collections import Counter
print('chunks carry metadata, e.g.:', {k: chunks[0][k] for k in ('id', 'topic')})
print('top topics:', Counter(c['topic'] for c in chunks).most_common(5))

# Flat FAISS has no native metadata filter: over-fetch, then filter in Python.
def retrieve_filtered(query, k=3, topic=None):
    qv = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True,
                         convert_to_numpy=True).astype('float32')
    sc, ii = index.search(qv, 50)
    hits = [{**chunks[int(i)], 'score': float(s)} for s, i in zip(sc[0], ii[0])]
    if topic: hits = [h for h in hits if h['topic'] == topic]
    return hits[:k]

q = 'how do I control access to the cluster?'
hits = retrieve(q, 8)
TOPIC = Counter(h['topic'] for h in hits).most_common(1)[0][0]   # a topic present in the results
print(f'\nno filter        :', [h['topic'] for h in hits])
print(f'topic={TOPIC!r:<11}:', [h['topic'] for h in retrieve_filtered(q, 8, topic=TOPIC)])

## 2 · Hybrid search (vector ∪ BM25)

Dense vectors capture *meaning* but can under-rank **exact terms** (component names, error codes). **BM25** is keyword search. We fuse the two rankings with **Reciprocal Rank Fusion** — meaning *and* literal matches. But hybrid is **not a free win**: it helps exact-term queries, yet on a semantically-paired Q&A set it can *hurt* (we measure this in §4).

In [ ]:
from rank_bm25 import BM25Okapi
bm25 = BM25Okapi([c['text'].lower().split() for c in chunks])

def hybrid(query, k=5, depth=30):
    # dense ranking
    qv = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True,
                         convert_to_numpy=True).astype('float32')
    _, di = index.search(qv, depth); dense = [int(i) for i in di[0]]
    # sparse (BM25) ranking
    bm = list(np.argsort(bm25.get_scores(query.lower().split()))[::-1][:depth])
    # Reciprocal Rank Fusion: fuse the two rankings
    rrf = {}
    for rank, i in enumerate(dense): rrf[i] = rrf.get(i, 0) + 1 / (60 + rank)
    for rank, i in enumerate(bm):    rrf[i] = rrf.get(i, 0) + 1 / (60 + rank)
    top = sorted(rrf, key=rrf.get, reverse=True)[:k]
    return [{**chunks[i], 'rrf': rrf[i]} for i in top]

q = 'what is a kubelet?'
print('dense-only top-3:', [h['topic'] for h in retrieve(q, 3)])
print('hybrid    top-3:', [h['topic'] for h in hybrid(q, 3)])

## 3 · Reranking

Vector/BM25 are fast but blunt. A **cross-encoder** reads query+chunk *together* and re-scores. Over-fetch cheaply, then keep the best few — the biggest lever after chunking.

In [ ]:
reranker = CrossEncoder('BAAI/bge-reranker-base', device='cpu')

# Over-fetch with hybrid, then the cross-encoder re-reads query+chunk together.
q = 'How does Kubernetes clean up unused container images?'
cands = hybrid(q, k=10)
print('BEFORE rerank (RRF order), top 3:')
for h in cands[:3]:
    print(f"  {h['topic']}: {h['text'][:70]}...")
scores = reranker.predict([[q, h['text']] for h in cands])
for h, s in zip(cands, scores): h['rr'] = float(s)
for h in sorted(cands, key=lambda h: h['rr'], reverse=True)[:3]:
    print('AFTER:', f"rr={h['rr']:.2f}  {h['topic']}: {h['text'][:60]}...")

## 4 · Does it actually help? — evaluate retrieval

Each question's own answer is its gold chunk, so we measure **recall@3** (gold chunk in the top 3?) and **MRR** (1/rank of the gold chunk). Compare three retrievers: vector-only (Lab 1) → + hybrid → vector + rerank (over a deep top-30). Two things to read: **reranking lifts MRR** (puts the right chunk first), and **hybrid here *hurts*** — proof that an advanced lever isn't automatically a win. Measure, don't assume.

In [ ]:
EVAL_N = 40          # raise for a tighter estimate (slower: reranks EVAL_N x depth pairs on CPU)
sample = chunks[:EVAL_N]

def rank_of(gold_id, ordered):
    for r, h in enumerate(ordered, 1):
        if h['id'] == gold_id: return r
    return None

def score(retriever):
    rr_sum = hit3 = 0
    for c in sample:
        r = rank_of(c['id'], retriever(c['question']))
        if r: rr_sum += 1 / r; hit3 += (r <= 3)
    return hit3 / len(sample), rr_sum / len(sample)

def vec(q):  return retrieve(q, 30)
def hyb(q):  return hybrid(q, 30)
def rerank_deep(q):
    cands = retrieve(q, 30)                       # over-fetch a deep pool, then rescore
    s = reranker.predict([[q, h['text']] for h in cands])
    for h, v in zip(cands, s): h['rr'] = float(v)
    return sorted(cands, key=lambda h: h['rr'], reverse=True)

print(f'over {EVAL_N} questions           recall@3    MRR')
for name, fn in [('vector-only', vec), ('+ hybrid (BM25)', hyb), ('vector + rerank (top-30)', rerank_deep)]:
    r, m = score(fn)
    print(f'  {name:<24} {r:5.2f}     {m:5.2f}')

## 5 · The full advanced answer (needs T4)

Query understanding (LLM multi-query) + hybrid → rerank → grounded, cited **Qwen2.5-3B** answer, plus a refusal. This loads the LLM (~6 GB) and **needs a T4** — on P100 it's skipped with a note, and the evaluation above still stands.

In [ ]:
# The grounded answer + LLM multi-query need the GPU LLM — this requires a **T4** runtime.
# Kaggle's P100 can't run the prebuilt fp16 kernels ('no kernel image'); the cell below
# is wrapped so the notebook still completes if you're on P100 — just switch to T4 x2.
try:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map='auto').eval()

    def generate(messages, max_new_tokens=200):
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inp = tok(text, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
        return tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()

    _ = generate([{'role': 'user', 'content': 'ok'}], max_new_tokens=2)   # surfaces a P100 CUDA error here

    # Query understanding: ask the LLM for alternate phrasings, union the hits (multi-query).
    def rewrites(query, n=3):
        msg = [{'role': 'user', 'content':
                f'Give {n} alternative search phrasings of this Kubernetes question, '
                f'one per line, no numbering:\n{query}'}]
        return [query] + [l.strip('-* ').strip() for l in generate(msg, 120).split(chr(10)) if l.strip()][:n]
    print('multi-query rephrasings of "pod stuck pending":')
    for r in rewrites('pod stuck pending'): print('  -', r)

    # Full advanced answer: hybrid -> rerank -> grounded, cited.
    SYS = ('You are a Kubernetes assistant. Use ONLY the CONTEXT to answer, and cite the chunk ids in [brackets]. '
           "Do not use any outside knowledge. If the answer is not in the CONTEXT, reply exactly: I don't know.")
    def answer(query, keep=3):
        cands = hybrid(query, k=20)
        s = reranker.predict([[query, h['text']] for h in cands])
        for h, v in zip(cands, s): h['rr'] = float(v)
        hits = sorted(cands, key=lambda h: h['rr'], reverse=True)[:keep]
        ctx = '\n'.join(f"[{h['id']}] {h['text']}" for h in hits)
        return generate([{'role': 'system', 'content': SYS},
                         {'role': 'user', 'content': f'CONTEXT:\n{ctx}\n\nQUESTION: {query}'}]), hits
    a, hits = answer('What is Role-Based Access Control (RBAC) in Kubernetes?')
    print('\nloaded:', [h['id'] for h in hits]); print('ANSWER:\n', textwrap.fill(a, 90))
    print('\nREFUSAL:', answer('What is the capital of France?')[0])
except Exception as e:
    print('LLM step skipped:', type(e).__name__, '-', str(e)[:120])
    print('>> Needs a T4 runtime: Settings -> Accelerator = GPU T4 x2 (not P100), then Run All.')

## Takeaways

- **Reranking lifts MRR more than recall** — here MRR rose (≈0.35 → 0.37) while recall@3
  barely moved: it doesn't find *more* right chunks, it puts the right one *first* — which
  is what the model actually reads.
- **Hybrid (BM25) backfired here** — it helps exact-term queries (the §2 demo), but on this
  semantically-paired Q&A set it *hurt* aggregate recall (0.38 → 0.25). An "advanced" lever
  is not a free win.
- **So: measure, don't assume.** recall@k / MRR tell you whether a lever helps *on your
  corpus* and *where* a failure is — "the demo answered well" does not.
- **Metadata filtering** stays a must for permissions/freshness regardless of the scores.
- Next: agentic RAG — the agent *decides* whether/what/when to retrieve (Layer 4).